# TikTok Audio Transcript PipelineThis notebook outlines a pipeline for fetching TikTok video data, downloading the audio from those videos, and transcribing the audio to text using OpenAI's Whisper.

### Section 1: Setup and Configuration
This section imports necessary libraries and defines key variables and paths for the project. It sets up the data directories and specifies the TikTok user handle to be analyzed.

In [1]:
import json, subprocess, time, random, shutil
from pathlib import Path
import pandas as pd

# The TikTok user handle to analyze
HANDLE = "humbletoker"
PROFILE_URL = f"https://www.tiktok.com/@{HANDLE}"

# Define directory paths for data storage
DATA_DIR = Path("../data")
AUDIO_DIR = DATA_DIR / "audio"
DERIVED_DIR = DATA_DIR / "derived"
AUDIO_DIR.mkdir(parents=True, exist_ok=True)
DERIVED_DIR.mkdir(parents=True, exist_ok=True)

# Specify the path to the ffmpeg executable
FFMPEG = shutil.which("ffmpeg") or "/Users/Logan/miniforge3/bin/ffmpeg"

# Define sleep intervals to be friendly to TikTok's servers
SLEEP_MIN, SLEEP_MAX = 0.8, 2.0  # rate-limit friendliness


### Section 2: Scrape Video Manifest from TikTok Profile
This section uses `yt-dlp` to scrape metadata for all videos on the specified TikTok profile. The metadata is then processed and saved into a CSV file.

In [2]:
# Use yt-dlp to dump video metadata as JSON
cmd = ["yt-dlp", "--dump-json", PROFILE_URL]
proc = subprocess.run(cmd, capture_output=True, text=True)
if proc.returncode != 0:
    print(proc.stderr[:4000])
    raise RuntimeError("yt-dlp profile scrape failed.")

# Process the JSON output to extract relevant video information
rows = []
for line in proc.stdout.splitlines():
    obj = json.loads(line)
    if obj.get("_type") == "playlist":
        continue
    vid = obj.get("id")
    url = obj.get("webpage_url") or obj.get("original_url")
    if not vid or not url:
        continue
    rows.append({
        "video_id": str(vid),
        "video_url": url,
        "title": obj.get("title"),
        "upload_date": obj.get("upload_date"),
        "view_count": obj.get("view_count"),
        "like_count": obj.get("like_count"),
        "comment_count": obj.get("comment_count"),
        "repost_count": obj.get("repost_count"),
    })

# Create a DataFrame and save the video manifest to a CSV file
videos = (
    pd.DataFrame(rows)
    .drop_duplicates("video_id")
    .sort_values("upload_date", ascending=False)
    .reset_index(drop=True)
)

manifest_path = DERIVED_DIR / "tiktok_manifest.csv"
videos.to_csv(manifest_path, index=False)
print("Videos found:", len(videos))
print("Saved manifest:", manifest_path.resolve())
videos.head()


### Section 3: Clear Existing Audio Files
This section clears out any previously downloaded audio files to ensure a fresh start.

In [3]:
# Remove all files in the audio directory
for p in AUDIO_DIR.glob("*"):
    if p.is_file():
        p.unlink()
print("Cleared:", AUDIO_DIR.resolve())


### Section 4: Download Audio from Videos
This section iterates through the video manifest and downloads the audio for each video using `yt-dlp`.

In [4]:
failed_ids = []

for _, r in videos.iterrows():
    vid = r["video_id"]
    url = r["video_url"]
    out_m4a = AUDIO_DIR / f"{vid}.m4a"
    if out_m4a.exists() and out_m4a.stat().st_size > 0:
        continue

    # Command to download audio using yt-dlp
    cmd = [
        "yt-dlp", url,
        "--no-playlist",
        "-x",
        "--audio-format", "m4a",
        "--audio-quality", "0",
        "-o", str(AUDIO_DIR / f"{vid}.%(ext)s"),
        "--force-overwrites",
        "--no-warnings",
    ]

    try:
        subprocess.run(cmd, check=True, capture_output=True, text=True)
    except subprocess.CalledProcessError:
        failed_ids.append(vid)

    # Sleep for a random interval to avoid being rate-limited
    time.sleep(random.uniform(SLEEP_MIN, SLEEP_MAX))

print("Audio downloaded:", len(list(AUDIO_DIR.glob("*.m4a"))))
print("Initial failures:", len(failed_ids))
failed_ids[:10]



### Section 5: Salvage Failed Downloads
This section attempts to download and extract audio for videos that failed in the previous step. This time, it downloads the video as an MP4 and then uses `ffmpeg` to extract the audio.

In [5]:
still_failed = []

for vid in failed_ids:
    url = f"https://www.tiktok.com/@{HANDLE}/video/{vid}"
    mp4_path = AUDIO_DIR / f"{vid}.mp4"
    m4a_path = AUDIO_DIR / f"{vid}.m4a"

    try:
        # Download the full video as an MP4
        subprocess.run([
            "yt-dlp", url,
            "--no-playlist",
            "-o", str(mp4_path),
            "--force-overwrites",
            "--no-warnings",
        ], check=True, capture_output=True, text=True)

        # Use ffmpeg to extract and convert the audio
        subprocess.run([
            FFMPEG, "-y",
            "-i", str(mp4_path),
            "-vn", # No video
            "-ac", "1", # Mono audio
            "-ar", "16000", # 16kHz sample rate
            str(m4a_path)
        ], check=True, capture_output=True, text=True)

        # Clean up the temporary MP4 file
        mp4_path.unlink(missing_ok=True)

    except subprocess.CalledProcessError:
        still_failed.append(vid)
        mp4_path.unlink(missing_ok=True)

print("After salvage, audio files:", len(list(AUDIO_DIR.glob("*.m4a"))))
print("Still failing:", len(still_failed))
still_failed


### Section 6: Transcribe Audio to Text
This section uses OpenAI's Whisper model to transcribe the downloaded audio files. The transcripts are then saved to Parquet and CSV files.

In [8]:
from pathlib import Path
import whisper
import warnings
warnings.filterwarnings(
    "ignore",
    message="FP16 is not supported on CPU; using FP32 instead",
    category=UserWarning,
)

# Load the Whisper model
model = whisper.load_model("base")  # bump to "small" for better accuracy if you want

audio_files = sorted(AUDIO_DIR.glob("*.m4a"))
print("Audio files to transcribe:", len(audio_files))

# Transcribe each audio file
rows = []
for p in audio_files:
    vid = p.stem
    result = model.transcribe(str(p))
    text = (result.get("text") or "").strip()
    rows.append({"video_id": vid, "transcript": text})

transcripts = pd.DataFrame(rows)

# Save the transcripts to Parquet and CSV files
out_parquet = DERIVED_DIR / "tiktok_transcripts.parquet"
out_csv = DERIVED_DIR / "tiktok_transcripts.csv"
transcripts.to_parquet(out_parquet, index=False)
transcripts.to_csv(out_csv, index=False)

print("Saved:", out_parquet.resolve())
print("Saved:", out_csv.resolve())
transcripts.head()


### Section 7: Inspect Transcripts
This section provides a summary of the generated transcripts and shows an example.

In [9]:
transcripts.describe()

In [12]:
# Display the transcript of the first video
transcripts.iloc[0]['transcript']